# 💵 Sahte Banknot Tespiti
## Veri Hazırlama ve Ön İşleme

**Marmara Üniversitesi | Yazılım Mühendisliği Temelleri**  
**Hazırlayan:** İbrahim Alo  
**Tarih:** 24 Kasım 2026  
**Proje ekibi:** Miraç Yaşar, Buğra Ahat, İbrahim Alo  
**Görev:** [Issue #4 – Model Eğitimi İçin Veri Hazırlama ve Ön İşleme](https://github.com/miracayasar/Sahte-Banknot-Tespiti/issues/4)

### Çalışmanın amacı
UCI Banknote Authentication veri setini Lojistik Regresyon ve Karar Ağacı modellerinin eğitimine uygun hâle getirmek. İki modelin aynı eğitim ve test kayıtlarını kullanması, karşılaştırmanın adil ve tekrar üretilebilir olmasını amaçlamaktadır.

### İş akışı
1. Veri yükleme, okuma ve temizleme
2. Özellikler ve hedef değişkenin ayrılması
3. Eğitim ve test kümelerinin oluşturulması
4. Lojistik Regresyon için standardizasyon
5. Doğrulama ve model eğitimine hazırlık

Bu notebook model eğitimi veya performans ölçümü içermez. Karar Ağacı için ölçeklendirilmemiş veriler ayrıca korunur.

---
## 1. Veri setinin yüklenmesi
Pandas, veri tablolarını okumak ve işlemek için kullanılacaktır. Google Colab'ın dosya yükleme aracı ile UCI'dan indirilen, içerisinde tek TXT dosyası bulunan ZIP arşivi çalışma ortamına aktarılacaktır.

- **Kaynak:** [UCI Banknote Authentication](https://archive.ics.uci.edu/dataset/267/banknote)
- **DOI:** [10.24432/C55P57](https://doi.org/10.24432/C55P57)
- **Lisans:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)

Ham veri dosyası dersin teslim kuralı gereği GitHub deposuna eklenmez. Notebook başka bir çalışma oturumunda açıldığında ZIP dosyası yeniden yüklenmelidir.

In [3]:

import pandas as pd
from google.colab import files

yuklenen = files.upload()


Saving banknote+authentication.zip to banknote+authentication (1).zip


### Sonuç
ZIP arşivi Colab çalışma ortamına başarıyla yüklenmiştir.

---
## 2. Veri setinin okunması
Pandas ile ZIP arşivindeki virgülle ayrılmış dosya okunacaktır. Dosya sütun başlıkları içermediğinden `header=None` kullanılır ve başlıklar açıkça atanır. İlk beş kayıt görüntülenerek tablo yapısı kontrol edilir.

| Değişken | Açıklama |
|---|---|
| `variance` | Dalgacık dönüşümlü görüntünün varyansı |
| `skewness` | Dalgacık dönüşümlü görüntünün çarpıklığı |
| `curtosis` | Dalgacık dönüşümlü görüntünün basıklığı |
| `entropy` | Görüntünün entropisi |
| `class` | İkili sınıf etiketi (0 veya 1) |

Veri seti ham banknot fotoğraflarını değil, önceden çıkarılmış sayısal özellikleri içermektedir.

In [4]:
dosya_adi = next(iter(yuklenen))

sutunlar = ["variance", "skewness", "curtosis", "entropy", "class"]

veri = pd.read_csv(dosya_adi, compression="zip", header=None, names=sutunlar)

veri.head()

,variance,skewness,curtosis,entropy,class
0,3.62160,8.6661,-2.8073,-0.44699,0
1,4.54590,8.1674,-2.4586,-1.46210,0
2,3.86600,-2.6383,1.9242,0.10645,0
3,3.45660,9.5228,-4.0112,-3.59440,0
4,0.32924,-4.4552,4.5718,-0.98880,0


### Sonuç
İlk beş kayıt, dört sayısal özellik ve bir sınıf etiketi içeren beş sütunlu bir tablo olarak görüntülenmiştir.

---
## 3. Veri kalitesinin kontrol edilmesi
Model eğitiminden önce veri setinin boyutu, her sütundaki eksik değer sayısı ve tam satır tekrarları kontrol edilecektir. `duplicated()` varsayılan olarak bir kaydın ilk örneğini değil, sonraki tekrarlarını işaretler.

In [5]:
print("Veri boyutu:", veri.shape)
print("Eksik değerler:")
print(veri.isna().sum())
print("Tekrarlanan satırlar:", veri.duplicated().sum())

Veri boyutu: (1372, 5)
Eksik değerler:
variance    0
skewness    0
curtosis    0
entropy     0
class       0
dtype: int64
Tekrarlanan satırlar: 24


### Bulgular ve yorum
| Kontrol | Sonuç |
|---|---:|
| Toplam kayıt | 1.372 |
| Sütun | 5 |
| Eksik değer | 0 |
| Tekrarlanan satır | 24 |

Eksik değer bulunmadığından doldurma işlemine gerek yoktur. Tespit edilen 24 tam satır tekrarı temizlenecektir.

---
## 4. Tekrarlanan kayıtların temizlenmesi
Tamamen aynı kayıtların eğitim ve test kümelerinde birden bulunması, ölçülen performansı olduğundan yüksek gösterebilir. Bu riski azaltmak için `drop_duplicates()` kullanılarak her tam satırın ilk örneği korunacaktır. Orijinal tablo değiştirilmeden temizlenmiş veri `temiz_veri` değişkeninde saklanacaktır.

In [6]:
temiz_veri = veri.drop_duplicates()

print("Temizlik öncesi:", veri.shape)
print("Temizlik sonrası:", temiz_veri.shape)
print("Kalan tekrarlar:", temiz_veri.duplicated().sum())

Temizlik öncesi: (1372, 5)
Temizlik sonrası: (1348, 5)
Kalan tekrarlar: 0


### Bulgular ve yorum
| Kontrol | Sonuç |
|---|---:|
| Temizlik öncesi | 1.372 |
| Kaldırılan tekrarlar | 24 |
| Temizlik sonrası | 1.348 |
| Kalan tam satır tekrarı | 0 |

Tam satır tekrarları kaldırılmış ve orijinal veri korunmuştur.

---
## 5. Veri türlerinin kontrol edilmesi
Dört girdi özelliğinin sayısal, hedef etiketinin ise tam sayı olması beklenmektedir. `dtypes` kullanılarak sütunların veri türleri kontrol edilecektir.

In [7]:
print(temiz_veri.dtypes)

variance    float64
skewness    float64
curtosis    float64
entropy     float64
class         int64
dtype: object


### Bulgular ve yorum
| Değişken | Veri türü |
|---|---|
| `variance` | `float64` |
| `skewness` | `float64` |
| `curtosis` | `float64` |
| `entropy` | `float64` |
| `class` | `int64` |

Sütun türleri uygundur; ek tür dönüşümü gerekmez. Etiketlerin gerçekten 0 ve 1 değerleriyle sınırlı olduğu son kontrol aşamasında ayrıca denetlenecektir.

---
## 6. Özelliklerin ve hedef değişkenin ayrılması
- **X (Features):** Modelin girdisi olan dört sayısal özellik.
- **y (Target):** Modelin tahmin edeceği sınıf etiketi.

Bu ayrım, hedef etiketinin girdi olarak modele verilmesini önler.

In [8]:
X = temiz_veri[["variance", "skewness", "curtosis", "entropy"]]
y = temiz_veri["class"]

print("X boyutu:", X.shape)
print("y boyutu:", y.shape)

X boyutu: (1348, 4)
y boyutu: (1348,)


### Bulgular ve yorum
| Değişken | Boyut |
|---|---|
| `X` | 1.348 × 4 |
| `y` | 1.348 |

Her kayıt için dört özellik ve bunlara karşılık gelen tek sınıf etiketi ayrılmıştır.

---
## 7. Eğitim ve test kümelerinin oluşturulması
Modelin daha önce görmediği veriler üzerindeki başarısını daha sonra ölçebilmek için temizlenmiş veri eğitim ve test kümelerine ayrılacaktır.

- `test_size=0.20`: Yaklaşık %80 eğitim, %20 test.
- `random_state=42`: Aynı veri sırası ve yazılım ortamında tekrarlanabilir veri ayrımı.
- `stratify=y`: Sınıf oranlarının iki kümede yaklaşık korunması.

Her iki model aynı kayıtları kullanacaktır. Test verisi model ayarlarını seçmek veya eğitim yapmak için kullanılmamalıdır.

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Eğitim boyutu:", X_train.shape)
print("Test boyutu:", X_test.shape)

Eğitim boyutu: (1078, 4)
Test boyutu: (270, 4)


### Bulgular ve yorum
| Küme | Hedef oran | Kayıt sayısı |
|---|---:|---:|
| Eğitim | %80 | 1.078 |
| Test | %20 | 270 |
| Toplam | %100 | 1.348 |

Ortak eğitim ve test kümeleri oluşturulmuştur. Kayıt sayıları, bölme sırasında yapılan tamsayı yuvarlaması nedeniyle oranlara yaklaşık karşılık gelir.

---
## 8. Özelliklerin ölçeklendirilmesi (Standardization)
Lojistik Regresyon için özelliklerin ölçeklerini karşılaştırılabilir hâle getirmek amacıyla `StandardScaler` uygulanacaktır. Dönüşüm her özellikten eğitim ortalamasını çıkarır ve eğitim standart sapmasına böler.

Veri sızıntısını önlemek için `fit_transform()` yalnızca eğitim verisinde kullanılacak; test verisine aynı öğrenilmiş dönüşüm `transform()` ile uygulanacaktır.

Karar Ağacı için standardizasyon gerekli değildir. Bu nedenle `X_train` ve `X_test` değiştirilmeden korunacaktır.

In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Sonuç
Lojistik Regresyon için `X_train_scaled` ve `X_test_scaled` oluşturulmuştur. Karar Ağacı için orijinal ölçekli kümeler korunmuştur. `scaler` nesnesi yeni girdilere aynı dönüşümü uygulamak için kullanılabilir.

---
## 9. Ölçeklendirmenin doğrulanması
Ölçeklendirilmiş eğitim verisinin her özelliğine ait ortalama ve standart sapma hesaplanacaktır. Dört özelliğin ortalamasının yaklaşık 0, standart sapmasının ise yaklaşık 1 olması beklenmektedir. Test kümesinde bu değerlerin aynen elde edilmesi gerekmez.

In [11]:
print("Ortalamalar:", X_train_scaled.mean(axis=0).round(4))
print("Standart sapmalar:", X_train_scaled.std(axis=0).round(4))

Ortalamalar: [ 0. -0.  0.  0.]
Standart sapmalar: [1. 1. 1. 1.]


### Bulgular ve yorum
| Özellik | Ortalama (yaklaşık) | Standart sapma |
|---|---:|---:|
| `variance` | 0 | 1 |
| `skewness` | 0 | 1 |
| `curtosis` | 0 | 1 |
| `entropy` | 0 | 1 |

Eğitim verisinde standardizasyon beklenen şekilde gerçekleşmiştir. Çıktıda görülen `-0.` hesaplama hassasiyeti ve yuvarlamadan kaynaklanır; `0.` ile sayısal olarak eşdeğerdir.

---
## 10. Genel veri kontrolü ve model eğitimine hazırlık
Temiz veri boyutu, eksik değerler, kalan tam satır tekrarları, eğitim ve test indekslerinin kesişimi ve iki modelin kullanacağı girdi/etiket boyutları son kez kontrol edilecektir.

In [12]:
print("=== GENEL VERİ KONTROLÜ ===")

print("Temiz kayıt sayısı:", len(temiz_veri))
print("Eksik değer sayısı:", temiz_veri.isna().sum().sum())
print("Tekrarlanan satır sayısı:", temiz_veri.duplicated().sum())
print("Ortak eğitim ve test indeksleri:",
      len(X_train.index.intersection(X_test.index)))

print("\n=== LOJİSTİK REGRESYON ===")

print("Eğitim girdileri:", X_train_scaled.shape)
print("Test girdileri:", X_test_scaled.shape)
print("Eğitim etiketleri:", y_train.shape)
print("Test etiketleri:", y_test.shape)

print("\n=== KARAR AĞACI ===")

print("Eğitim girdileri:", X_train.shape)
print("Test girdileri:", X_test.shape)
print("Eğitim etiketleri:", y_train.shape)
print("Test etiketleri:", y_test.shape)

=== GENEL VERİ KONTROLÜ ===
Temiz kayıt sayısı: 1348
Eksik değer sayısı: 0
Tekrarlanan satır sayısı: 0
Ortak eğitim ve test indeksleri: 0

=== LOJİSTİK REGRESYON ===
Eğitim girdileri: (1078, 4)
Test girdileri: (270, 4)
Eğitim etiketleri: (1078,)
Test etiketleri: (270,)

=== KARAR AĞACI ===
Eğitim girdileri: (1078, 4)
Test girdileri: (270, 4)
Eğitim etiketleri: (1078,)
Test etiketleri: (270,)


### Bulgular ve yorum
| Kontrol | Sonuç |
|---|---:|
| Temiz kayıt sayısı | 1.348 |
| Eksik değer | 0 |
| Kalan tam satır tekrarı | 0 |
| Ortak eğitim ve test indeksi | 0 |
| Eğitim girdileri (iki model) | 1.078 × 4 |
| Test girdileri (iki model) | 270 × 4 |
| Eğitim etiketleri | 1.078 |
| Test etiketleri | 270 |

Aynı satır indeksinin iki kümeye birden atanmadığı ve veri boyutlarının uyumlu olduğu görülmüştür. İndeks kontrolü tek başına aynı özellik değerlerine sahip farklı indeksleri tespit etmez; bu nedenle aşağıda ek bir doğrulama yapılmıştır.

---
## 11. Ek bütünlük ve tekrar üretilebilirlik kontrolü
Sınıf etiketlerinin beklenen değerleri içerdiği, sayısal değerlerin sonlu olduğu, özellik ve etiket indekslerinin uyumlu olduğu ve aynı özellik vektörünün iki kümeye birden düşmediği doğrulanmıştır. Ayrıca eğitim/test sınıf dağılımları ve kullanılan kütüphane sürümleri aşağıdaki çıktıda gösterilmiştir.

**Not:** Ek bütünlük ve tekrar üretilebilirlik kontrolleri, orijinal veri seti üzerinde çalıştırılmış ve başarıyla tamamlanmıştır. Bir `assert` kontrolü başarısız olursa model eğitimine geçmeden önce nedenini inceleyin.

In [13]:
import numpy as np
import sklearn

# Etiket ve sayısal değer kontrolü
assert set(y.unique()) == {0, 1}, "Sınıf etiketlerini kontrol edin."
assert np.isfinite(X.to_numpy()).all(), "Sonlu olmayan özellik değeri var."

# Girdi-etiket eşleşmesi ve veri ayrımının kontrolü
assert X_train.index.equals(y_train.index), "Eğitim indeksleri uyuşmuyor."
assert X_test.index.equals(y_test.index), "Test indeksleri uyuşmuyor."
assert len(X_train.index.intersection(X_test.index)) == 0
assert len(X_train) + len(X_test) == len(temiz_veri)

# Farklı indekslerdeki aynı özellik vektörlerinin kontrolü
ortak_ozellikler = X_train.drop_duplicates().merge(
    X_test.drop_duplicates(), on=list(X.columns), how="inner"
)
print("Ortak özellik vektörü sayısı:", len(ortak_ozellikler))
assert ortak_ozellikler.empty, "Aynı özellikler iki kümede bulunuyor."

# Standardizasyonun ve boyutların kontrolü
assert X_train_scaled.shape == X_train.shape
assert X_test_scaled.shape == X_test.shape
assert np.isfinite(X_train_scaled).all() and np.isfinite(X_test_scaled).all()
assert np.allclose(X_train_scaled.mean(axis=0), 0)
assert np.allclose(X_train_scaled.std(axis=0), 1)
assert np.allclose(X_test_scaled, scaler.transform(X_test))

print("\nSınıf dağılımları (kayıt sayısı):")
print(pd.DataFrame({
    "Eğitim": y_train.value_counts(),
    "Test": y_test.value_counts()
}).sort_index())
print("\nPandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("NumPy:", np.__version__)
print("\nTüm ek kontroller başarılı.")

Ortak özellik vektörü sayısı: 0

Sınıf dağılımları (kayıt sayısı):
       Eğitim  Test
class              
0         590   148
1         488   122

Pandas: 2.2.3
Scikit-learn: 1.6.1
NumPy: 2.4.6

Tüm ek kontroller başarılı.


---
## 12. Genel değerlendirme ve ekip üyeleri için kullanım rehberi

Veri kalitesi kontrol edilmiş, 24 tam satır tekrarı kaldırılmış, 1.348 kayıt üzerinde ortak eğitim-test ayrımı yapılmış ve Lojistik Regresyon için standardizasyon uygulanmıştır.Son ek doğrulama hücresi başarıyla çalıştırılmış ve veri hazırlama sürecinin model eğitimine uygunluğu doğrulanmıştır.

### Model eğitiminde kullanılacak değişkenler
| Veri | Lojistik Regresyon | Karar Ağacı |
|---|---|---|
| Eğitim girdileri | `X_train_scaled` | `X_train` |
| Test girdileri | `X_test_scaled` | `X_test` |
| Eğitim etiketleri | `y_train` | `y_train` |
| Test etiketleri | `y_test` | `y_test` |

### Çalıştırma ve paylaşma
1. [UCI kaynağından](https://archive.ics.uci.edu/dataset/267/banknote) orijinal Banknote Authentication ZIP arşivini indirin (tek TXT dosyası içeren arşiv).
2. Bu `.ipynb` dosyasını Google Colab'da açın. İlk hücreyi çalıştırıp ZIP dosyasını yükleyin.
3. Bütün kod hücrelerini yukarıdan aşağıya çalıştırın. Ek kontrollerin başarılı olduğunu doğrulayın.
4. Model geliştirme kodunu aynı çalışma oturumunda, hazırlık hücrelerinin ardından çalıştırın. Ayrı notebook kullanılacaksa bu hazırlık adımlarını aynı veri dosyası ve ayarlarla yeniden çalıştırın. Değişkenler ayrı Colab oturumlarına kendiliğinden taşınmaz.
5. Model karşılaştırması için veri temizleme yöntemini, kayıt sırasını ve veri ayrımı parametrelerini değiştirmeyin. Sürümleri kaydedin; gerekirse aynı ayrımı ekipçe karşılaştırın.

**Model ayarlarının seçimi:** İleride çapraz doğrulama ile Lojistik Regresyon ayarları seçilirse, standardizasyon her doğrulama katında yalnızca ilgili eğitim bölümünde öğrenilmelidir (örneğin `Pipeline` ile). Nihai test kümesi en son değerlendirme için saklanmalıdır. Yeni banknot girdilerinde de eğitimden öğrenilmiş dönüşüm kullanılmalıdır.

### Kapsam ve sonraki adımlar
Bu çalışma veri ön işleme ile sınırlıdır. Model eğitimi, tahmin, doğruluk ve diğer performans ölçütlerinin hesaplanması sonraki proje aşamalarında yapılacaktır. Burada herhangi bir model başarı oranı iddia edilmemektedir.

**Proje deposu:** [Sahte-Banknot-Tespiti](https://github.com/miracayasar/Sahte-Banknot-Tespiti)

**Veri seti atfı:** Lohweg, V. (2012). *Banknote Authentication* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C55P57

---
**Marmara Üniversitesi | Yazılım Mühendisliği Temelleri | 2026–2027 Güz Dönemi**